# Q10 — How sensitive is BPMA to the CV design?

Hypothesis: BPMA inherits the validity of its validation design. Use matched stratified splitters here as a starter; extend the experiment to grouped, blocked, and rolling splitters when the estimator API can receive the required group or time information.

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
while root != root.parent and not (root / 'bayesian_predictive_model_averaging').is_dir():
    root = root.parent
sys.path.insert(0, str(root))
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold
from EXPERIMENTS.common import classification_data, split_data, fit_classifier

In [2]:
X, y = classification_data(seed=37, kind='nonlinear')
X_train, X_test, y_train, y_test = split_data(X, y, seed=37)
splitters = {
    'stratified-3': StratifiedKFold(n_splits=3, shuffle=True, random_state=37),
    'stratified-5': StratifiedKFold(n_splits=5, shuffle=True, random_state=37),
}
results = {}
for label, splitter in splitters.items():
    model = fit_classifier(X_train, y_train, seed=37, cv=splitter)
    results[label] = {
        'test_log_loss': log_loss(y_test, model.predict_proba(X_test)),
        'family_mass': model.get_model_masses()['family'],
    }
results

{'stratified-3': {'test_log_loss': 0.43019141365943675,
  'family_mass': {'gaussian_mixture': 0.38030552095996384,
   'knn': 0.11581930088679758,
   'linear_mixture': 0.2539450401310146,
   'mlp': 0.2499301380222239}},
 'stratified-5': {'test_log_loss': 0.4609128108835218,
  'family_mass': {'gaussian_mixture': 0.3767179507289578,
   'knn': 0.1258022106237067,
   'linear_mixture': 0.24975053404079067,
   'mlp': 0.2477293046065448}}}

To answer the scientific question, generate group and time-ordered synthetic data and add splitter support that respects those structures. Include random-fold leakage as a negative control and evaluate on held-out groups or future observations.

## Conclusion from the executed starter run

**Status: not falsified; validation-design sensitivity observed.** Stratified 3-fold CV produced test log loss 0.4302, compared with 0.4609 for stratified 5-fold, while family masses remained broadly similar. The experiment now confirms that custom splitter objects execute correctly, but it covers only exchangeable binary data; grouped, temporal, and leakage controls remain necessary.